In [1]:
import pandas as pd
from IPython.display import display
from sklearn.model_selection import LeaveOneOut
from sklearn.preprocessing import MinMaxScaler
import torch
from torch.utils.data import DataLoader, TensorDataset
from tqdm import tqdm
import matplotlib.pyplot as plt
import numpy as np
import torch
import pandas as pd
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import LeaveOneOut
from torch.utils.data import DataLoader, TensorDataset
from tqdm import tqdm

In [2]:
import pandas as pd
import numpy as np
import torch
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import KFold
from sklearn.preprocessing import MinMaxScaler
from tqdm import tqdm

# Load CSV and split features and labels
df = pd.read_csv('null-corrected.csv', index_col=0)
print(df.head())

# Convert the last column into a binary target:
# If value > 0, the target is 1 (True); if value == 0, the target is 0 (False)
Y = (df.iloc[:, -1].astype(int) > 0).astype(int)
X = df.iloc[:, :-1]

print(X.head())
print(Y.head(10))

# Preprocess features: scale to [-1, 1]
scaler_X = MinMaxScaler(feature_range=(-1, 1))
X_scaled = scaler_X.fit_transform(X)

# Convert labels to a torch tensor for binary classification.
# Using dtype float32 and unsqueeze(1) to have shape (N, 1) for a single output.
Y_tensor = torch.tensor(Y.values, dtype=torch.float32).unsqueeze(1)

# Set device: use CUDA if available otherwise CPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

# Define input dimension (output now is a single value for binary classification)
input_dim = X_scaled.shape[1]

# Outer CV: 10-fold split using KFold
outer_kf = KFold(n_splits=10, shuffle=True, random_state=42)

# List of regularization candidates (weight decay values)
regularizations = [0.001, 0.005, 0.01, 0.05, 0.1]

results = []  # Will store the best regularization and test loss for each outer fold
fold_idx = 0

# Outer loop over the 10 folds
for outer_train_index, outer_test_index in tqdm(outer_kf.split(X_scaled), desc="Outer KFold Folds"):
    fold_idx += 1

    # Split outer training and test data
    X_outer_train, X_outer_test = X_scaled[outer_train_index], X_scaled[outer_test_index]
    Y_outer_train, Y_outer_test = Y_tensor[outer_train_index], Y_tensor[outer_test_index]

    # Convert outer training and test data to torch tensors
    X_outer_train_tensor = torch.tensor(X_outer_train, dtype=torch.float32)
    Y_outer_train_tensor = Y_outer_train.clone().detach()
    X_outer_test_tensor = torch.tensor(X_outer_test, dtype=torch.float32).to(device)
    Y_outer_test_tensor = Y_outer_test.clone().detach().to(device)

    # Inner CV: 5-fold split on the outer training data
    inner_kf = KFold(n_splits=5, shuffle=True, random_state=42)
    best_inner_avg_loss = float('inf')
    best_reg = None

    # Loop over each regularization candidate
    for reg in regularizations:
        inner_losses = []

        # Inner loop: 5 folds for hyperparameter selection
        for inner_train_idx, inner_val_idx in inner_kf.split(X_outer_train):
            # Define inner train and validation sets
            X_inner_train = X_outer_train[inner_train_idx]
            X_inner_val = X_outer_train[inner_val_idx]
            Y_inner_train = Y_outer_train[inner_train_idx]
            Y_inner_val = Y_outer_train[inner_val_idx]

            # Convert inner train and validation sets to torch tensors
            X_inner_train_tensor = torch.tensor(X_inner_train, dtype=torch.float32)
            Y_inner_train_tensor = Y_inner_train.clone().detach()
            X_inner_val_tensor = torch.tensor(X_inner_val, dtype=torch.float32).to(device)
            Y_inner_val_tensor = Y_inner_val.clone().detach().to(device)

            # Create DataLoader for inner training data
            inner_train_dataset = TensorDataset(X_inner_train_tensor, Y_inner_train_tensor)
            inner_train_loader = DataLoader(
                inner_train_dataset, batch_size=32, shuffle=True,
                pin_memory=True if device.type == 'cuda' else False
            )

            # Initialize a fresh model for the inner fold.
            # Adjusted model to output a single value.
            model_inner = torch.nn.Sequential(
                torch.nn.Linear(input_dim, 1)
            ).to(device)
            
            # Use BCEWithLogitsLoss for binary classification.
            criterion = torch.nn.BCEWithLogitsLoss()
            optimizer = torch.optim.Adam(model_inner.parameters(), lr=0.01, weight_decay=reg)

            epochs = 10  # set as needed
            # Train on the inner training set
            for epoch in range(epochs):
                model_inner.train()
                for inputs, labels in inner_train_loader:
                    inputs = inputs.to(device, non_blocking=True)
                    labels = labels.to(device, non_blocking=True)

                    optimizer.zero_grad()
                    outputs = model_inner(inputs)
                    loss = criterion(outputs, labels)
                    loss.backward()
                    optimizer.step()

            # Evaluate the model on the inner validation set
            model_inner.eval()
            with torch.no_grad():
                outputs_val = model_inner(X_inner_val_tensor)
                loss_val = criterion(outputs_val, Y_inner_val_tensor)
                inner_losses.append(loss_val.item())

        # Average loss for this candidate over the inner folds
        avg_inner_loss = np.mean(inner_losses)
        if avg_inner_loss < best_inner_avg_loss:
            best_inner_avg_loss = avg_inner_loss
            best_reg = reg

    # After inner CV, retrain the model on the entire outer training set with the best regularization
    outer_train_dataset = TensorDataset(X_outer_train_tensor, Y_outer_train_tensor)
    outer_train_loader = DataLoader(
        outer_train_dataset, batch_size=32, shuffle=True,
        pin_memory=True if device.type == 'cuda' else False
    )
    
    model_outer = torch.nn.Sequential(
        torch.nn.Linear(input_dim, 1)
    ).to(device)
    
    criterion_outer = torch.nn.BCEWithLogitsLoss()
    optimizer_outer = torch.optim.Adam(model_outer.parameters(), lr=0.01, weight_decay=best_reg)

    for epoch in range(10):  # adjust epochs as needed
        model_outer.train()
        for inputs, labels in outer_train_loader:
            inputs = inputs.to(device, non_blocking=True)
            labels = labels.to(device, non_blocking=True)

            optimizer_outer.zero_grad()
            outputs = model_outer(inputs)
            loss = criterion_outer(outputs, labels)
            loss.backward()
            optimizer_outer.step()

    # Evaluate the retrained model on the outer test set
    model_outer.eval()
    with torch.no_grad():
        outputs_test = model_outer(X_outer_test_tensor)
        loss_test = criterion_outer(outputs_test, Y_outer_test_tensor).item()

    print(f"Fold {fold_idx}: Selected Best Reg: {best_reg}, Outer Test Loss: {loss_test:.4f}")
    results.append({'fold': fold_idx, 'regularization': best_reg, 'test_loss': loss_test})

# Print a summary of results from all folds
results_df = pd.DataFrame(results)
print("\nResults for each fold:")
print(results_df)

mean_loss = results_df['test_loss'].mean()
print(f"\nKFold Mean Test Loss: {mean_loss:.4f}")


/home/botlaplinux/anaconda3/envs/ai/lib/python3.13/site-packages/torch/cuda/__init__.py:129: UserWarning: CUDA initialization: CUDA unknown error - this may be due to an incorrectly set up environment, e.g. changing env variable CUDA_VISIBLE_DEVICES after program start. Setting the available devices to be zero. (Triggered internally at /pytorch/c10/cuda/CUDAFunctions.cpp:109.)
  return torch._C._cuda_getDeviceCount() > 0


    age  sex   cp  trestbps   chol  fbs  restecg  thalach  exang  oldpeak  \
0  63.0  1.0  1.0     145.0  233.0  1.0      2.0    150.0    0.0      2.3   
1  67.0  1.0  4.0     160.0  286.0  0.0      2.0    108.0    1.0      1.5   
2  67.0  1.0  4.0     120.0  229.0  0.0      2.0    129.0    1.0      2.6   
3  37.0  1.0  3.0     130.0  250.0  0.0      0.0    187.0    0.0      3.5   
4  41.0  0.0  2.0     130.0  204.0  0.0      2.0    172.0    0.0      1.4   

   slope   ca  thal  num  
0    3.0  0.0   6.0    0  
1    2.0  3.0   3.0    2  
2    2.0  2.0   7.0    1  
3    3.0  0.0   3.0    0  
4    1.0  0.0   3.0    0  
    age  sex   cp  trestbps   chol  fbs  restecg  thalach  exang  oldpeak  \
0  63.0  1.0  1.0     145.0  233.0  1.0      2.0    150.0    0.0      2.3   
1  67.0  1.0  4.0     160.0  286.0  0.0      2.0    108.0    1.0      1.5   
2  67.0  1.0  4.0     120.0  229.0  0.0      2.0    129.0    1.0      2.6   
3  37.0  1.0  3.0     130.0  250.0  0.0      0.0    187.0    0.0   

Outer KFold Folds: 1it [00:01,  1.72s/it]

Fold 1: Selected Best Reg: 0.01, Outer Test Loss: 0.3348


Outer KFold Folds: 2it [00:02,  1.30s/it]

...

Fold 8: Selected Best Reg: 0.001, Outer Test Loss: 0.4401


Outer KFold Folds: 9it [00:09,  1.01it/s]

Fold 9: Selected Best Reg: 0.001, Outer Test Loss: 0.4143


Outer KFold Folds: 10it [00:10,  1.05s/it]

Fold 10: Selected Best Reg: 0.01, Outer Test Loss: 0.4180

Results for each fold:
   fold  regularization  test_loss
0     1           0.010   0.334821
1     2           0.001   0.359600
2     3           0.001   0.496215
3     4           0.001   0.304011
4     5           0.001   0.388364
5     6           0.010   0.342405
6     7           0.005   0.549473
7     8           0.001   0.440119
8     9           0.001   0.414255
9    10           0.010   0.417998

KFold Mean Test Loss: 0.4047
